In [1]:
import pandas as pd
import certifi
from neo4j import GraphDatabase, TrustCustomCAs
import json
import pprint

# Sostituisci questi segnaposto con le tue credenziali Neo4j Aura
# Il tuo URI di Aura sarà simile a neo4j+s://xxxxxxx.databases.neo4j.io
NEO4J_URI      = "neo4j://78976b17.databases.neo4j.io"
NEO4J_USER     = "78976b17"
NEO4J_PASSWORD = "RbsKnY-5Wf0aTTdOfqC60d-dTlVVf-LzTpXmvRcfJXc"
NEO4J_DATABASE = "78976b17" # O il nome del tuo database se diverso

# Nota: usiamo lo schema "neo4j" (invece di "neo4j+s") con encrypted=True e
# trusted_certificates=TrustCustomCAs(certifi.where()) perché su alcune macchine
# (es. Windows con store dei certificati non aggiornato) il driver non trova la
# CA root "SSL.com" usata dal certificato di Aura nello store di sistema, pur
# essendo il certificato del server valido. Forzare l'uso del bundle certifi
# (che include già quella CA) risolve l'errore "Unable to retrieve routing
# information" / "self-signed certificate in certificate chain".

# Ora possiamo tentare la connessione
try:
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
        encrypted=True,
        trusted_certificates=TrustCustomCAs(certifi.where()),
    )
    driver.verify_connectivity()
    print(f"Connesso a {NEO4J_URI}")
except Exception as e:
    print(f"Errore di connessione a Neo4j: {e}")

def run(cypher, **params):
    """Cypher → pandas DataFrame."""
    with driver.session(database=NEO4J_DATABASE) as s:
        return pd.DataFrame([r.data() for r in s.run(cypher, **params)])

def run_write(cypher, **params):
    """Run a write query and return its summary counters."""
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.execute_write(
            lambda tx: tx.run(cypher, **params).consume().counters
        )

Connesso a neo4j://78976b17.databases.neo4j.io


In [ ]:
df = pd.read_csv("aree_specialistiche_lombardia.csv")
df.head(5)

,Area
0,ALLERGOLOGIA
1,ANALISI DI LABORATORIO
2,ANATOMIA PATOLOGICA
3,ANGIOLOGIA
4,ASTANTERIA - DEGENZA BREVE


In [126]:
run_write('MATCH (n) DETACH DELETE n')

query = '''
LOAD CSV WITH HEADERS FROM "https://drive.google.com/uc?export=download&id=1i0eor8Utt4zFKPoHQnfbJrldwxzlAev0" AS row FIELDTERMINATOR ',' // Replace with a URL accessible by Neo4j Aura
WITH row // Added WITH row to pass data to the MERGE clause
MERGE (rep:Reparto {Nome: row.Area})
'''

run_write(query)

SummaryCounters({'contains-updates': True, 'labels-added': 83, 'nodes-created': 83, 'properties-set': 83})

In [127]:
with open("ospedali_docs.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

pprint.pprint(docs)

[{'Aree_specialistiche': [{'Nome': 'ANATOMIA PATOLOGICA'},
                          {'Nome': 'CARDIOCHIRURGIA'},
                          {'Nome': 'CARDIOCHIRURGIA PEDIATRICA'},
                          {'Nome': 'CARDIOLOGIA'},
                          {'Nome': 'CHIRURGIA GENERALE'},
                          {'Nome': 'CHIRURGIA MAXILLO FACCIALE'},
                          {'Nome': 'CHIRURGIA PEDIATRICA'},
                          {'Nome': 'CHIRURGIA PLASTICA E MEDICINA ESTETICA'},
                          {'Nome': 'CHIRURGIA TORACICA'},
                          {'Nome': 'CHIRURGIA VASCOLARE'},
                          {'Nome': 'CURE PALLIATIVE - HOSPICE'},
                          {'Nome': 'DERMATOLOGIA'},
                          {'Nome': 'DIETOLOGIA E NUTRIZIONE CLINICA'},
                          {'Nome': 'ELETTROFISIOLOGIA E EMODINAMICA'},
                          {'Nome': 'EMATOLOGIA E IMMUNOEMATOLOGIA'},
                          {'Nome': 'EMODIALISI'},
            

In [128]:
run_write("""
CREATE CONSTRAINT Codice IF NOT EXISTS
FOR (o:Ospedale)
REQUIRE o.Codice IS UNIQUE
""")

SummaryCounters({})

In [129]:
query = '''
UNWIND $docs AS row

WITH row
WHERE row.Classificazione = 'HUB'
MERGE (o:Ospedale:Hub {Codice: row.Codice})

SET
o.Nome = row.Nome,
o.Indirizzo = row.Indirizzo,
o.CAP = row.CAP,
o.Città = row.Città,
o.Ente = row.Ente,
o.Coordinate = point({
    latitude: row.Coordinate.lat,
    longitude: row.Coordinate.long
})

'''

run_write(query, docs=docs)

SummaryCounters({'contains-updates': True, 'labels-added': 22, 'nodes-created': 11, 'properties-set': 77})

In [130]:
query = '''
UNWIND $docs AS row

WITH row
WHERE row.Classificazione <> 'HUB'
MERGE (o:Ospedale:Spoke {Codice: row.Codice})

SET
o.Nome = row.Nome,
o.Indirizzo = row.Indirizzo,
o.CAP = row.CAP,
o.Città = row.Città,
o.Ente = row.Ente,
o.Coordinate = point({
    latitude: row.Coordinate.lat,
    longitude: row.Coordinate.long
})

'''

run_write(query, docs=docs)

SummaryCounters({'contains-updates': True, 'labels-added': 172, 'nodes-created': 86, 'properties-set': 620})

In [131]:
query = '''
UNWIND $docs AS row

MATCH (o:Ospedale {Codice: row.Codice})

UNWIND row.Aree_specialistiche AS area

MATCH (r:Reparto {Nome: area.Nome})

MERGE (o)-[:HA_REPARTO]->(r)
'''

run_write(query, docs=docs)

SummaryCounters({'contains-updates': True, 'relationships-created': 2723})

In [132]:
query = '''
MATCH (o:Ospedale)-[:HA_REPARTO]->(r:Reparto)
WHERE r.Nome CONTAINS "PRONTO SOCCORSO" AND r.Nome <> "PRONTO SOCCORSO"

RETURN o.Nome, o.Città, r.Nome'''

run(query)

,o.Nome,o.Città,r.Nome
0,IRCCS OSPEDALE SAN RAFFAELE,MILANO,PRONTO SOCCORSO CARDIOLOGICO
1,IRCCS MULTIMEDICA - SESTO SAN GIOVANNI,SESTO SAN GIOVANNI,PRONTO SOCCORSO CARDIOLOGICO
2,ISTITUTO CLINICO HUMANITAS - MATER DOMINI,CASTELLANZA,PRONTO SOCCORSO CARDIOLOGICO
3,CENTRO CARDIOLOGICO MONZINO,MILANO,PRONTO SOCCORSO CARDIOLOGICO
4,OSPEDALE SAN LUCA,MILANO,PRONTO SOCCORSO CARDIOLOGICO
...,...,...,...
65,OSPEDALE FILIPPO DEL PONTE,VARESE,PRONTO SOCCORSO PEDIATRICO
66,OSPEDALE DI TRADATE,TRADATE,PRONTO SOCCORSO PEDIATRICO
67,OSPEDALE DI BUSTO ARSIZIO,BUSTO ARSIZIO,PRONTO SOCCORSO PEDIATRICO
68,OSPEDALE DI GALLARATE,GALLARATE,PRONTO SOCCORSO PEDIATRICO


In [133]:
query = '''
MATCH (o1:Ospedale), (o2:Ospedale)

WHERE elementId(o1) < elementId(o2)

WITH o1,o2,
point.distance(o1.Coordinate,o2.Coordinate) AS distanza

MERGE (o1)-[r:DISTA]->(o2)

SET r.km = round(distanza/1000.0,2)
'''

run_write(query)

SummaryCounters({'contains-updates': True, 'relationships-created': 4656, 'properties-set': 4656})

In [134]:
# Duomo di Milano:
# latitude: 45.463966,
# longitude: 9.190578

# Casa di Dario:
# latitude: 45.3083,
# longitude: 10.1497

# Posto dimenticato da Dio in provincia di Sondrio:
# latitude: 46.44349,
# longitude: 9.33406

query = '''
MERGE (p:Persona)

SET p.Coordinate = point({
    latitude: 45.463966,
    longitude: 9.190578
})

'''

run_write(query, docs=docs)

SummaryCounters({'contains-updates': True, 'labels-added': 1, 'nodes-created': 1, 'properties-set': 1})

In [135]:
query = '''
MATCH (p:Persona), (o:Ospedale)

WITH p,o,
point.distance(p.Coordinate,o.Coordinate) AS distanza

MERGE (p)-[r:DISTA_DA_OSPEDALE]->(o)

SET r.km = round(distanza/1000.0,2)
'''

run_write(query)

SummaryCounters({'contains-updates': True, 'relationships-created': 97, 'properties-set': 97})

In [136]:
query = '''
MATCH p=(:Persona)-[r:DISTA_DA_OSPEDALE]->(o:Ospedale)-[:HA_REPARTO]->(rp:Reparto)
WHERE rp.Nome CONTAINS "UROLOGIA E ANDROLOGIA"
ORDER BY r.km
RETURN o.Nome as Ospedale, o.Città as Città, rp.Nome as Reparto, r.km as Distanza LIMIT 5
'''

run(query)

,Ospedale,Città,Reparto,Distanza
0,OSPEDALE FATEBENEFRATELLI E OFTALMICO,MILANO,UROLOGIA E ANDROLOGIA,1.72
1,OSPEDALE SANT'AMBROGIO,MILANO,UROLOGIA E ANDROLOGIA,1.80
2,POLICLINICO DI MILANO,MILANO,UROLOGIA E ANDROLOGIA,2.28
3,ISTITUTO EUROPEO DI ONCOLOGIA (IEO),MILANO,UROLOGIA E ANDROLOGIA,2.30
4,IRCCS ISTITUTO NAZIONALE TUMORI,MILANO,UROLOGIA E ANDROLOGIA,3.38


In [137]:
query = '''
MATCH (p:Persona)-[r:DISTA_DA_OSPEDALE]->(o:Ospedale)-[:HA_REPARTO]->(rep:Reparto)

WITH rep, o, r
ORDER BY r.km ASC

WITH rep, collect({
    ospedale: o,
    distanza: r.km
})[0] AS migliore

RETURN
rep.Nome AS reparto,
migliore.ospedale.Nome AS ospedale,
migliore.distanza AS km

ORDER BY reparto'''

df = run(query)

df.head()

,reparto,ospedale,km
0,ALLERGOLOGIA,OSPEDALE SANT'AMBROGIO,1.80
1,ANALISI DI LABORATORIO,OSPEDALE SANT'AMBROGIO,1.80
2,ANATOMIA PATOLOGICA,OSPEDALE FATEBENEFRATELLI E OFTALMICO,1.72
3,ANGIOLOGIA,ISTITUTO EUROPEO DI ONCOLOGIA (IEO),2.30
4,ASTANTERIA - DEGENZA BREVE,OSPEDALE FATEBENEFRATELLI E OFTALMICO,1.72
